# MosaicMRI chunk category and protocol counts

This notebook scans the split tar chunks under `/data2/MosaicMRI/chunks`, ignores the challenge folders, and counts anatomy/category plus protocol labels using the filename maps in `mosaicmri/category_maps`.

It reads tar headers only. H5 file payloads are skipped by seeking through the split archive.

In [1]:
from pathlib import Path
import bisect
import json
import tarfile
from collections import Counter, defaultdict
from html import escape
from IPython.display import HTML, display

ROOT = Path('/home/paula')
CHUNKS_DIR = Path('/data2/MosaicMRI/chunks')
CATEGORY_MAP = ROOT / 'mosaicmri/category_maps/file_category_mapping.json'
PROTOCOL_MAP = ROOT / 'mosaicmri/category_maps/file_protocol_mapping.json'

IGNORE_SPLITS = {
    'anatomy_transfer_challenge__ankle',
    'contrast_generalization_challenge__T1_FS',
}

category_by_file = json.loads(CATEGORY_MAP.read_text())
protocol_by_file = json.loads(PROTOCOL_MAP.read_text())

split_dirs = sorted(
    p for p in CHUNKS_DIR.iterdir()
    if p.is_dir() and p.name not in IGNORE_SPLITS
)

[(p.name, len(list(p.glob('*.tar.part-*')))) for p in split_dirs]

[('multicoil_test', 7), ('multicoil_train', 27), ('multicoil_val', 7)]

In [2]:
class SplitTarReader:
    """Seekable reader over tar parts, without concatenating huge files."""

    def __init__(self, parts):
        self.parts = [Path(p) for p in parts]
        self.sizes = [p.stat().st_size for p in self.parts]
        self.starts = []
        total = 0
        for size in self.sizes:
            self.starts.append(total)
            total += size
        self.total = total
        self.pos = 0
        self._idx = None
        self._fh = None

    def tell(self):
        return self.pos

    def seekable(self):
        return True

    def readable(self):
        return True

    def close(self):
        if self._fh is not None:
            self._fh.close()
            self._fh = None
            self._idx = None

    def _open_idx(self, idx):
        if idx < 0 or idx >= len(self.parts):
            return None
        if self._idx != idx:
            if self._fh is not None:
                self._fh.close()
            self._fh = open(self.parts[idx], 'rb')
            self._idx = idx
        return self._fh

    def seek(self, offset, whence=0):
        if whence == 0:
            new_pos = offset
        elif whence == 1:
            new_pos = self.pos + offset
        elif whence == 2:
            new_pos = self.total + offset
        else:
            raise ValueError(f'unsupported whence: {whence}')
        if new_pos < 0:
            raise ValueError('negative seek')
        self.pos = min(new_pos, self.total)
        return self.pos

    def read(self, n=-1):
        if self.pos >= self.total:
            return b''
        if n is None or n < 0:
            n = self.total - self.pos
        n = min(n, self.total - self.pos)
        out = bytearray()
        while n > 0 and self.pos < self.total:
            idx = bisect.bisect_right(self.starts, self.pos) - 1
            rel = self.pos - self.starts[idx]
            fh = self._open_idx(idx)
            fh.seek(rel)
            take = min(n, self.sizes[idx] - rel)
            chunk = fh.read(take)
            if not chunk:
                break
            out.extend(chunk)
            self.pos += len(chunk)
            n -= len(chunk)
        return bytes(out)


def part_for_offset(starts, offset):
    return bisect.bisect_right(starts, offset) - 1


def scan_split(split_dir):
    parts = sorted(split_dir.glob('*.tar.part-*'))
    reader = SplitTarReader(parts)
    rows = []
    try:
        with tarfile.open(fileobj=reader, mode='r:') as tf:
            for member in tf:
                if not member.isfile():
                    continue
                idx = part_for_offset(reader.starts, member.offset)
                filename = Path(member.name).name
                rows.append({
                    'split': split_dir.name,
                    'chunk': parts[idx].name,
                    'filename': filename,
                    'tar_path': member.name,
                    'size_bytes': member.size,
                    'header_offset': member.offset,
                    'category': category_by_file.get(filename),
                    'protocol': protocol_by_file.get(filename),
                })
    finally:
        reader.close()
    return rows

In [3]:
file_rows = []
for split_dir in split_dirs:
    file_rows.extend(scan_split(split_dir))

len(file_rows)

2542

In [4]:
def count_rows(file_rows, field):
    counts = Counter(
        (row['split'], row['chunk'], row[field] or '<missing>')
        for row in file_rows
    )
    return [
        {'split': split, 'chunk': chunk, field: value, 'count': count}
        for (split, chunk, value), count in sorted(counts.items())
    ]

category_counts = count_rows(file_rows, 'category')
protocol_counts = count_rows(file_rows, 'protocol')

missing_summary = {
    'files_scanned': len(file_rows),
    'missing_category': sum(row['category'] is None for row in file_rows),
    'missing_protocol': sum(row['protocol'] is None for row in file_rows),
}
missing_summary

{'files_scanned': 2542, 'missing_category': 18, 'missing_protocol': 0}

In [5]:
def rows_to_html(rows, title=None, max_rows=None):
    shown = rows if max_rows is None else rows[:max_rows]
    if not shown:
        return HTML('<p>No rows.</p>')
    columns = list(shown[0].keys())
    html = []
    if title:
        html.append(f'<h3>{escape(title)}</h3>')
    html.append('<table>')
    html.append('<thead><tr>' + ''.join(f'<th>{escape(str(c))}</th>' for c in columns) + '</tr></thead>')
    html.append('<tbody>')
    for row in shown:
        html.append('<tr>' + ''.join(f'<td>{escape(str(row.get(c, "")))}</td>' for c in columns) + '</tr>')
    html.append('</tbody></table>')
    if max_rows is not None and len(rows) > max_rows:
        html.append(f'<p>Showing {max_rows} of {len(rows)} rows.</p>')
    return HTML('\n'.join(html))


def pivot_counts(counts, value_field):
    labels = sorted({row[value_field] for row in counts})
    grouped = defaultdict(dict)
    for row in counts:
        grouped[(row['split'], row['chunk'])][row[value_field]] = row['count']
    pivot = []
    for split, chunk in sorted(grouped):
        out = {'split': split, 'chunk': chunk}
        total = 0
        for label in labels:
            value = grouped[(split, chunk)].get(label, 0)
            out[label] = value
            total += value
        out['total'] = total
        pivot.append(out)
    return pivot

category_pivot = pivot_counts(category_counts, 'category')
protocol_pivot = pivot_counts(protocol_counts, 'protocol')

## Anatomy/category counts per split and chunk

In [6]:
display(rows_to_html(category_pivot, 'Category pivot'))

split,chunk,<missing>,ANKLE,ELBOW,FOOT,HIP,KNEE,LEG_TIBFIB,PELVIS_PUBALGIA,SCAPULA_RIBS_FEMUR,SHOULDER,SPINE,WRIST_HAND_THUMB,total
multicoil_test,multicoil_test.tar.part-00000,0,5,1,1,3,15,1,2,0,7,33,2,70
multicoil_test,multicoil_test.tar.part-00001,0,4,0,2,7,12,1,0,0,8,30,0,64
multicoil_test,multicoil_test.tar.part-00002,0,6,1,2,3,6,3,0,1,12,24,2,60
multicoil_test,multicoil_test.tar.part-00003,0,1,0,1,6,5,2,0,0,11,30,3,59
multicoil_test,multicoil_test.tar.part-00004,0,3,1,2,5,10,1,1,3,13,25,0,64
multicoil_test,multicoil_test.tar.part-00005,0,1,1,2,6,8,0,1,1,7,48,2,77
multicoil_test,multicoil_test.tar.part-00006,0,0,0,0,1,0,0,0,0,0,5,0,6
multicoil_train,multicoil_train.tar.part-00000,0,0,0,1,3,19,0,0,0,11,41,0,75
multicoil_train,multicoil_train.tar.part-00001,0,0,3,10,6,6,0,3,0,6,35,6,75
multicoil_train,multicoil_train.tar.part-00002,0,0,4,2,2,19,0,1,0,5,43,5,81


## Protocol counts per split and chunk

In [7]:
display(rows_to_html(protocol_pivot, 'Protocol pivot'))

split,chunk,PD,PD_FS,STIR,T1,T1_FS,T2,T2_FS,total
multicoil_test,multicoil_test.tar.part-00000,4,18,10,15,2,18,3,70
multicoil_test,multicoil_test.tar.part-00001,5,13,3,13,4,24,2,64
multicoil_test,multicoil_test.tar.part-00002,0,10,8,12,7,15,8,60
multicoil_test,multicoil_test.tar.part-00003,2,10,2,10,1,27,7,59
multicoil_test,multicoil_test.tar.part-00004,3,8,7,12,2,19,13,64
multicoil_test,multicoil_test.tar.part-00005,4,7,8,18,1,31,8,77
multicoil_test,multicoil_test.tar.part-00006,0,1,2,1,0,2,0,6
multicoil_train,multicoil_train.tar.part-00000,3,12,8,17,0,27,8,75
multicoil_train,multicoil_train.tar.part-00001,1,12,8,16,0,25,13,75
multicoil_train,multicoil_train.tar.part-00002,4,11,10,20,0,29,7,81


## PELVIS_PUBALGIA chunk order CSV

This final cell writes a CSV ordered by chunks with the most `PELVIS_PUBALGIA` files and includes the filenames in each chunk.

In [8]:
import csv

PELVIS_CATEGORY = 'PELVIS_PUBALGIA'
pelvis_by_chunk = defaultdict(list)

for row in file_rows:
    if row['category'] == PELVIS_CATEGORY:
        pelvis_by_chunk[(row['split'], row['chunk'])].append(row)

pelvis_pubalgia_chunk_rows = []
for (split, chunk), rows in pelvis_by_chunk.items():
    rows = sorted(rows, key=lambda r: r['header_offset'])
    pelvis_pubalgia_chunk_rows.append({
        'split': split,
        'chunk': chunk,
        'pelvis_pubalgia_count': len(rows),
        'filenames': ';'.join(row['filename'] for row in rows),
    })

pelvis_pubalgia_chunk_rows = sorted(
    pelvis_pubalgia_chunk_rows,
    key=lambda row: (-row['pelvis_pubalgia_count'], row['split'], row['chunk']),
)

pelvis_csv_path = ROOT / 'mosaicmri/dataset_chunks/pelvis_pubalgia_chunks_ordered.csv'
with pelvis_csv_path.open('w', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['split', 'chunk', 'pelvis_pubalgia_count', 'filenames'],
    )
    writer.writeheader()
    writer.writerows(pelvis_pubalgia_chunk_rows)

print(f'Wrote {len(pelvis_pubalgia_chunk_rows)} rows to {pelvis_csv_path}')
display(rows_to_html(pelvis_pubalgia_chunk_rows, 'PELVIS_PUBALGIA chunks ordered by count'))

Wrote 17 rows to /home/paula/mosaicmri/dataset_chunks/pelvis_pubalgia_chunks_ordered.csv


split,chunk,pelvis_pubalgia_count,filenames
multicoil_train,multicoil_train.tar.part-00005,5,meas_MID00136_FID115648_RT_COR_T2_TSE_FS.h5;meas_MID00141_FID116612_COR_T1_WP.h5;meas_MID00145_FID116616_LT_SAG_PD_FS.h5;meas_MID00150_FID116621_LT_AX_PD_FS_UNI.h5;meas_MID00154_FID116625_COR_OBL_T2_FS.h5
multicoil_train,multicoil_train.tar.part-00004,4,meas_MID00109_FID115621_COR_T1_WP.h5;meas_MID00119_FID115631_RT_SAG_PD_FS.h5;meas_MID00121_FID115633_LT_SAG_PD_FS.h5;meas_MID00128_FID115640_AX_T1.h5
multicoil_train,multicoil_train.tar.part-00018,4,meas_MID00355_FID135471_COR_T1_WP.h5;meas_MID00367_FID135483_UNI_SAG_PD_FS_RT.h5;meas_MID00369_FID135485_UNI_SAG_PD_FS_LT.h5;meas_MID00370_FID135486_AX_T1.h5
multicoil_train,multicoil_train.tar.part-00020,4,meas_MID00398_FID135514_UNI_SAG_PD_FS_RT.h5;meas_MID00400_FID135516_UNI_SAG_PD_FS_LT.h5;meas_MID00405_FID135521_AX_T1.h5;meas_MID00413_FID135529_COR_OBL_T1.h5
multicoil_train,multicoil_train.tar.part-00001,3,meas_MID00052_FID123988_COR_T1_WP.h5;meas_MID00058_FID123994_AX_T2_FS_WP.h5;meas_MID00063_FID123999_LT_SAG_PD_FS.h5
multicoil_test,multicoil_test.tar.part-00000,2,meas_MID00252_FID130322_COR_T1_WP.h5;meas_MID00259_FID130329_AX_OBL_PD_FS.h5
multicoil_train,multicoil_train.tar.part-00017,2,meas_MID00343_FID123709_AX_T2_FS_WP.h5;meas_MID00353_FID123719_AX_T1.h5
multicoil_test,multicoil_test.tar.part-00004,1,meas_MID00254_FID130324_AX_OBL_T2_FS.h5
multicoil_test,multicoil_test.tar.part-00005,1,meas_MID00251_FID130321_SAG_PD_FS_SYM_PUB.h5
multicoil_train,multicoil_train.tar.part-00002,1,meas_MID00068_FID124004_AX_T1.h5
